In [35]:
# pytorch 8

# optuna : hyperparams tuning

# complete fMNIST dataset (70000 images)

In [36]:
import pandas as pd
import numpy as np

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset,DataLoader
import torchmetrics

import optuna

In [37]:
# !pip install torchmetrics

# !pip install optuna

In [38]:
df = pd.read_csv('/content/drive/MyDrive/fmnist_full_dataset/fashion-mnist_train.csv')

df.head()

,label,pixel1,pixel2,pixel3,pixel4,pixel5,pixel6,pixel7,pixel8,pixel9,...,pixel775,pixel776,pixel777,pixel778,pixel779,pixel780,pixel781,pixel782,pixel783,pixel784
0,2,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,9,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,6,0,0,0,0,0,0,0,5,0,...,0,0,0,30,43,0,0,0,0,0
3,0,0,0,0,1,2,0,0,0,0,...,3,0,0,0,0,1,0,0,0,0
4,3,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


In [39]:
X = df.drop('label',axis=1).values
y = df['label'].values

In [40]:
# scaling

X = X/255.0

In [41]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
device

device(type='cuda')

In [42]:
# CustomDataset class

class CustomDataset(Dataset):
  def __init__(self,x,y):
    self.x = torch.tensor(x,dtype=torch.float32)
    self.y = torch.tensor(y,dtype=torch.long)

  def __len__(self):
    return len(self.x)

  def __getitem__(self, idx):
    return self.x[idx],self.y[idx]

In [43]:
dataset = CustomDataset(X,y)

In [44]:
# network dynamic arch

class Network(nn.Module):

  def __init__(self,
               input_dim,
               output_dim,
               n_hidden_layers,
               n_neurons,
               dropout_rate):

    super().__init__()

    layers = []

    for i in range(n_hidden_layers):
      layers.append(nn.Linear(input_dim,n_neurons))
      layers.append(nn.BatchNorm1d(n_neurons))
      layers.append(nn.ReLU())
      layers.append(nn.Dropout(dropout_rate))
      input_dim = n_neurons

    layers.append(nn.Linear(input_dim,output_dim))


    self.model = nn.Sequential(*layers)

  def forward(self,x):
    return self.model(x)

In [45]:
# train - val split

from sklearn.model_selection import train_test_split

X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

In [46]:
# optuna objective

def objective(trial):

    n_hidden_layers = trial.suggest_int('n_hidden_layers', 1, 3)
    n_neurons = trial.suggest_int('n_neurons', 64, 512, step=64)
    optimizer_name = trial.suggest_categorical('optimizer', ['Adam', 'SGD'])
    learning_rate = trial.suggest_float('learning_rate', 1e-4, 1e-2, log=True)
    dropout_rate = trial.suggest_float('dropout_rate', 0.1, 0.4)
    batch_size = trial.suggest_categorical('batch_size', [64, 128])
    weight_decay = trial.suggest_float('weight_decay', 1e-5,1e-1, log=True)
    n_epochs = trial.suggest_int('n_epochs', 10, 100,step=10)

    train_ds = CustomDataset(X_train, y_train)
    val_ds = CustomDataset(X_val, y_val)

    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True)
    val_loader = DataLoader(val_ds, batch_size=batch_size)

    model = Network(784, 10, n_hidden_layers, n_neurons, dropout_rate).to(device)

    optimizer = getattr(optim, optimizer_name)(model.parameters(), lr=learning_rate,weight_decay=weight_decay)

    criterion = nn.CrossEntropyLoss()

    accuracy_metric = torchmetrics.Accuracy(task="multiclass", num_classes=10).to(device)

    for epoch in range(20): # Fixed epochs for tuning efficiency
        model.train()
        for batch_X, batch_y in train_loader:
            batch_X, batch_y = batch_X.to(device), batch_y.to(device)
            optimizer.zero_grad()
            loss = criterion(model(batch_X), batch_y)
            loss.backward()
            optimizer.step()

        # Validation and Pruning
        model.eval()
        accuracy_metric.reset()
        with torch.no_grad():
            for batch_X, batch_y in val_loader:
                preds = torch.argmax(model(batch_X.to(device)), dim=1)
                accuracy_metric.update(preds, batch_y.to(device))

        current_acc = accuracy_metric.compute().item()

        # Report back to Optuna for pruning
        trial.report(current_acc, epoch)
        if trial.should_prune():
            raise optuna.exceptions.TrialPruned()

    return current_acc

In [47]:
study = optuna.create_study(direction='maximize')

study.optimize(objective,n_trials=10)

print(f"Best params : {study.best_params}")
print(f"Best Accuracy : {study.best_value}")

[I 2026-02-02 17:41:16,670] A new study created in memory with name: no-name-3ab888bb-c4b7-417a-bfdc-3c4303922596
[I 2026-02-02 17:41:48,183] Trial 0 finished with value: 0.89041668176651 and parameters: {'n_hidden_layers': 3, 'n_neurons': 448, 'optimizer': 'Adam', 'learning_rate': 0.0002726193004786152, 'dropout_rate': 0.17704956811647854, 'batch_size': 128, 'weight_decay': 0.0015827395817276635, 'n_epochs': 30}. Best is trial 0 with value: 0.89041668176651.
[I 2026-02-02 17:42:43,495] Trial 1 finished with value: 0.8492500185966492 and parameters: {'n_hidden_layers': 3, 'n_neurons': 448, 'optimizer': 'Adam', 'learning_rate': 0.0001733487227054665, 'dropout_rate': 0.1764332455570216, 'batch_size': 64, 'weight_decay': 0.06199828817893783, 'n_epochs': 50}. Best is trial 0 with value: 0.89041668176651.
[I 2026-02-02 17:43:04,769] Trial 2 finished with value: 0.8100000023841858 and parameters: {'n_hidden_layers': 1, 'n_neurons': 384, 'optimizer': 'SGD', 'learning_rate': 0.0003651142502880

Best params : {'n_hidden_layers': 3, 'n_neurons': 448, 'optimizer': 'Adam', 'learning_rate': 0.0002726193004786152, 'dropout_rate': 0.17704956811647854, 'batch_size': 128, 'weight_decay': 0.0015827395817276635, 'n_epochs': 30}
Best Accuracy : 0.89041668176651


In [48]:
# recreating network with best params

best_params = study.best_params

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
criterion = nn.CrossEntropyLoss()

model = Network(
    input_dim=784,
    output_dim=10,
    n_hidden_layers=best_params['n_hidden_layers'],
    n_neurons=best_params['n_neurons'],
    dropout_rate=best_params['dropout_rate']
).to(device)

In [49]:
# optimizer with best params

if best_params['optimizer'] == 'adam':
    optimizer = optim.Adam(
        model.parameters(),
        lr=best_params['learning_rate'],
        weight_decay=best_params['weight_decay']
    )
elif best_params['optimizer'] == 'sgd':
    optimizer = optim.SGD(
        model.parameters(),
        lr=best_params['learning_rate'],
        weight_decay=best_params['weight_decay']
    )
else:
    optimizer = optim.RMSprop(
        model.parameters(),
        lr=best_params['learning_rate'],
        weight_decay=best_params['weight_decay']
    )

In [50]:
# retaining with best params

full_train_ds = CustomDataset(X, y)

train_loader = DataLoader(
    full_train_ds,
    batch_size=best_params['batch_size'],
    shuffle=True
)

for epoch in range(best_params['n_epochs']):
    model.train()
    total_loss = 0

    for batch_X, batch_y in train_loader:
        batch_X = batch_X.view(-1, 784).to(device)
        batch_y = batch_y.to(device)

        optimizer.zero_grad()
        logits = model(batch_X)
        loss = criterion(logits, batch_y)
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    print(f"Epoch {epoch+1} Loss: {total_loss/len(train_loader):.4f}")

Epoch 1 Loss: 0.4602
Epoch 2 Loss: 0.3569
Epoch 3 Loss: 0.3305
Epoch 4 Loss: 0.3115
Epoch 5 Loss: 0.2992
Epoch 6 Loss: 0.2883
Epoch 7 Loss: 0.2815
Epoch 8 Loss: 0.2741
Epoch 9 Loss: 0.2653
Epoch 10 Loss: 0.2587
Epoch 11 Loss: 0.2575
Epoch 12 Loss: 0.2500
Epoch 13 Loss: 0.2466
Epoch 14 Loss: 0.2410
Epoch 15 Loss: 0.2345
Epoch 16 Loss: 0.2311
Epoch 17 Loss: 0.2281
Epoch 18 Loss: 0.2234
Epoch 19 Loss: 0.2207
Epoch 20 Loss: 0.2177
Epoch 21 Loss: 0.2130
Epoch 22 Loss: 0.2125
Epoch 23 Loss: 0.2079
Epoch 24 Loss: 0.2079
Epoch 25 Loss: 0.2042
Epoch 26 Loss: 0.2015
Epoch 27 Loss: 0.2000
Epoch 28 Loss: 0.1958
Epoch 29 Loss: 0.1930
Epoch 30 Loss: 0.1919


In [51]:
# loading test data

test_df = pd.read_csv("/content/drive/MyDrive/fmnist_full_dataset/fashion-mnist_test.csv")

In [52]:
X_test = test_df.drop('label',axis=1).values
y_test = test_df['label'].values

In [53]:
X_test = X_test/255.0

In [54]:
test_ds = CustomDataset(X_test, y_test)

test_loader = DataLoader(test_ds, batch_size=best_params['batch_size'])

In [55]:
from torchmetrics import Accuracy

accuracy = Accuracy(task="multiclass", num_classes=10).to(device)

model.eval()

accuracy.reset()

with torch.no_grad():
    for batch_X, batch_y in test_loader:
        batch_X = batch_X.view(-1, 784).to(device)
        batch_y = batch_y.to(device)

        logits = model(batch_X)
        preds = torch.argmax(logits, dim=1)
        accuracy.update(preds, batch_y)

test_acc = accuracy.compute().item()

print(f"\nTest Accuracy: {test_acc:.4f}")


Test Accuracy: 0.8932


In [56]:
# saving model

torch.save(model.state_dict(), "best_fmnist_model.pth")

In [57]:
# loading model

# model.load_state_dict(torch.load("best_fmnist_model.pth"))
# model.eval()